In [6]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageOps

# Check for RTX 3050
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")

✅ Using device: cuda


In [7]:
# 0 is strictly reserved for the CTC Blank token
ALPHABET = "0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ!\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~ "
char_to_int = {char: i + 1 for i, char in enumerate(ALPHABET)}
int_to_char = {i + 1: char for i, char in enumerate(ALPHABET)}
NUM_CLASSES = len(ALPHABET) + 1 

print(f"✅ Vocabulary initialized with {NUM_CLASSES} classes.")

✅ Vocabulary initialized with 96 classes.


In [8]:
class WordDataset(Dataset):
    def __init__(self, txt_file, img_dir):
        self.img_dir = img_dir
        self.data = []
        with open(txt_file, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip(): continue
                parts = line.strip().split()
                if len(parts) >= 9 and parts[1] == 'ok':
                    filename = parts[0] + ".png" 
                    text = " ".join(parts[8:]) 
                    if len(text) <= 16: # Keep sequences manageable for the CNN width
                        self.data.append((filename, text))
        self.transform = transforms.ToTensor()

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        filename, text = self.data[idx]
        img_path = os.path.join(self.img_dir, filename)
        try:
            im = Image.open(img_path).convert('L')
        except FileNotFoundError:
            im = Image.new('L', (128, 32), color=255)
            text = ""
        # Preprocessing for CNN: Height 32, Width 128
        new_w = int(im.size[0] * (32 / im.size[1]))
        im = im.resize((new_w, 32), Image.LANCZOS)
        if new_w < 128:
            im = ImageOps.expand(im, (0, 0, 128 - new_w, 0), fill=255)
        else:
            im = im.crop((0, 0, 128, 32))
        return self.transform(im), torch.tensor([char_to_int[c] for c in text if c in char_to_int], dtype=torch.long)

def collate_fn(batch):
    images, targets = zip(*batch)
    images = torch.stack(images, 0)
    target_lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)
    targets = torch.cat(targets) 
    return images, targets, target_lengths

In [9]:
class HandwrittenWordReader(nn.Module):
    def __init__(self, num_chars, hidden_size=256, num_layers=2):
        super(HandwrittenWordReader, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d((2, 2), (2, 1), (0, 1)) 
        )
        self.rnn = nn.LSTM(256 * 4, hidden_size, num_layers, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_size * 2, num_chars)

    def forward(self, x):
        conv_out = self.cnn(x)
        b, c, h, w = conv_out.size()
        conv_out = conv_out.view(b, c * h, w).permute(0, 2, 1) 
        rnn_out, _ = self.rnn(conv_out)
        return F.log_softmax(self.fc(rnn_out), dim=2).permute(1, 0, 2)

model = HandwrittenWordReader(num_chars=NUM_CLASSES).to(device)

In [ ]:
# Paths for your local AryanPC setup
word_train_ds = WordDataset('iam_data/words_new.txt', 'iam_data/iam_words/words')
train_loader = DataLoader(word_train_ds, batch_size=32, shuffle=True, collate_fn=collate_fn)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0002)
criterion = nn.CTCLoss(blank=0, zero_infinity=True) 

for epoch in range(50):
    model.train()
    running_loss = 0.0
    for images, targets, target_lengths in train_loader:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        preds = model(images)
        input_lengths = torch.full((images.size(0),), preds.size(0), dtype=torch.long)
        loss = criterion(preds, targets, input_lengths, target_lengths)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0) # Circuit breaker
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}/50 | Loss: {running_loss/len(train_loader):.4f}")

torch.save(model.state_dict(), 'crnn_word_reader_final.pth')

Epoch 1/50 | Loss: 0.7329
